In [3]:
import csv
import os
from datetime import datetime


In [4]:
class Movie:

    def __init__(self, movie_id, title, genre, duration, showtime, price):
        self.movie_id = movie_id
        self.title = title
        self.genre = genre
        self.duration = duration
        self.showtime = showtime
        self.price = price



movie1 = Movie(
    "M101",
    "Interstellar",
    "Sci-Fi",
    169,
    ("10:00 AM", "2:30 PM", "6:25 PM"),
    200
)

movie2 = Movie(
    "M102",
    "Inception",
    "Thriller",
    148,
    ("11:00 AM", "3:15 PM", "7:00 PM"),
    180
)

movie3 = Movie(
    "M103",
    "Avengers: Infinity War",
    "Action",
    181,
    ("12:00 PM", "4:00 PM", "8:00 PM"),
    250
)

movie4 = Movie(
    "M104",
    "The Dark Knight",
    "Action",
    152,
    ("1:00 PM", "5:00 PM", "9:00 PM"),
    220
)



movies = [movie1, movie2, movie3, movie4]


for movie in movies:
    print(movie.movie_id, movie.title, movie.genre)


M101 Interstellar Sci-Fi
M102 Inception Thriller
M103 Avengers: Infinity War Action
M104 The Dark Knight Action


In [5]:
class Booking:

    def __init__(self, booking_id, customer_name, phone, movie_title,
                 timing, seat, ticket_type, price, booking_time):

        self.booking_id = booking_id
        self.customer_name = customer_name
        self.phone = phone
        self.movie_title = movie_title
        self.timing = timing
        self.seat = seat
        self.ticket_type = ticket_type
        self.price = price
        self.booking_time = booking_time


    def to_dict(self):

        return {
            "booking_id": self.booking_id,
            "customer_name": self.customer_name,
            "phone": self.phone,
            "movie_title": self.movie_title,
            "timing": self.timing,
            "seat": self.seat,
            "ticket_type": self.ticket_type,
            "price": self.price,
            "booking_time": self.booking_time
        }


    def display(self):

        print("\n---------- BOOKING ----------")
        print("Booking ID :", self.booking_id)
        print("Customer   :", self.customer_name)
        print("Phone      :", self.phone)
        print("Movie      :", self.movie_title)
        print("Show       :", self.timing)
        print("Seat       :", self.seat)
        print("Ticket     :", self.ticket_type)
        print(f"Amount     : ₹{float(self.price):.2f}")
        print("Booked On  :", self.booking_time)


In [6]:
class BookingSystem:

    FILE_NAME = "bookings.csv"

    FIELD_NAMES = [
        "booking_id", "customer_name", "phone", "movie_title",
        "timing", "seat", "ticket_type", "price", "booking_time"
    ]


    def __init__(self, movies):

        self.movies = movies
        self.bookings = []

        self.seats = {
            "A1", "A2", "A3", "A4", "A5",
            "B1", "B2", "B3", "B4", "B5",
            "C1", "C2", "C3", "C4", "C5",
            "D1", "D2", "D3", "D4", "D5"
        }

        self.booked_seats = {}

        self.booking_counter = 1

        self.load_bookings_from_file()




    def view_movies(self):

        print("\n-------- movies airing -------------")

        for movie in self.movies:

            print(f"ID : {movie.movie_id}")
            print(f"Title : {movie.title}")
            print(f"Genre : {movie.genre}")
            print(f"Duration : {movie.duration} minutes")
            print(f"Show Timings : {movie.showtime}")
            print(f"Price : ₹{movie.price}")
            print("-" * 60)


    def search_movie(self):

        print("\n---------- SEARCH MOVIE ----------")
        print("1. Search by Title")
        print("2. Search by Genre")

        choice = input("Enter choice: ").strip()

        results = []

        if choice == "1":

            keyword = input("Enter title (or part of it): ").strip().lower()

            for movie in self.movies:

                if keyword in movie.title.lower():
                    results.append(movie)

        elif choice == "2":

            keyword = input("Enter genre: ").strip().lower()

            for movie in self.movies:

                if keyword in movie.genre.lower():
                    results.append(movie)

        else:

            print("Invalid choice.")
            return

        if not results:

            print("No matching movies found.")

        else:

            print(f"\nFound {len(results)} movie(s):")

            for movie in results:

                print(f"{movie.movie_id} - {movie.title} ({movie.genre})")



    def select_movie_and_show(self):

        print("\n---------- selection of movie ----------")

        movie_id = input("Enter Movie ID: ").strip().upper()

        for movie in self.movies:

            if movie.movie_id == movie_id:

                print(f"\nSelected Movie: {movie.title}")
                print("\nAvailable Show Timings:")

                for index, timing in enumerate(movie.showtime, start=1):
                    print(f"{index}. {timing}")

                while True:

                    try:

                        choice = int(input("Select show timing: "))

                        if 1 <= choice <= len(movie.showtime):

                            selected_timing = movie.showtime[choice - 1]

                            return movie, selected_timing

                        else:

                            print(
                                "Invalid choice. "
                                "Please select a valid timing."
                            )

                    except ValueError:

                        print("Please enter a number.")

        print("Movie ID not found.")

        return None, None




    def show_seat_map(self, movie, timing):

        show_key = (movie.movie_id, timing)

        if show_key not in self.booked_seats:

            self.booked_seats[show_key] = set()

        booked = self.booked_seats[show_key]

        print("\nAvailable Seats:  ( X = already booked )")

        for row in ["A", "B", "C", "D"]:

            for number in range(1, 6):

                seat = f"{row}{number}"

                if seat in booked:

                    print(" X ", end=" ")

                else:

                    print(f"{seat}", end=" ")

            print()

        return booked


    def view_seat_map_menu(self):

        print("\n---------- VIEW SEAT MAP ----------")

        movie, timing = self.select_movie_and_show()

        if movie is not None:

            print(f"\nMovie: {movie.title}   Show: {timing}")

            self.show_seat_map(movie, timing)


    def select_seat(self, movie, timing):

        print("\n---------- selection of seats ----------")

        booked = self.show_seat_map(movie, timing)

        while True:

            seat_choice = input(
                "\nEnter your seat: "
            ).strip().upper()

            if seat_choice not in self.seats:

                print("Invalid seat number. Please try again.")

            elif seat_choice in booked:

                print("Sorry, that seat is already booked.")

            else:

                booked.add(seat_choice)

                print(
                    f"Seat {seat_choice} "
                    "selected successfully!"
                )

                return seat_choice



    def get_customer_details(self):

        print("\n---------- CUSTOMER DETAILS ----------")

        while True:

            name = input("Enter customer name: ").strip()

            if name:
                break

            print("Name cannot be empty. Please try again.")


        while True:

            phone = input("Enter phone number: ").strip()

            if phone.isdigit() and len(phone) == 10:
                break

            print("Please enter a valid 10-digit phone number.")


        return name, phone



    def select_ticket_type(self):

        print("\n---------- TICKET TYPE ----------")

        ticket_types = {
            "1": ("Regular", 1.0),
            "2": ("Premium", 1.5),
            "3": ("VIP", 2.0)
        }

        print("1. Regular")
        print("2. Premium")
        print("3. VIP")

        while True:

            choice = input("Select ticket type: ").strip()

            if choice in ticket_types:

                ticket_name, multiplier = ticket_types[choice]

                return ticket_name, multiplier

            else:

                print("Invalid choice. Please select 1, 2, or 3.")


    def calculate_price(self, movie, multiplier):

        final_price = movie.price * multiplier

        return final_price




    def generate_booking_id(self):

        booking_id = f"BK{self.booking_counter:04d}"

        self.booking_counter += 1

        return booking_id




    def load_bookings_from_file(self):

        if not os.path.exists(self.FILE_NAME):
            return

        try:

            with open(self.FILE_NAME, "r", newline="") as file:

                reader = csv.DictReader(file)

                for row in reader:

                    booking = Booking(
                        row["booking_id"],
                        row["customer_name"],
                        row["phone"],
                        row["movie_title"],
                        row["timing"],
                        row["seat"],
                        row["ticket_type"],
                        float(row["price"]),
                        row["booking_time"]
                    )

                    self.bookings.append(booking)

                    show_key = None

                    for movie in self.movies:

                        if movie.title == booking.movie_title:

                            show_key = (movie.movie_id, booking.timing)

                    if show_key is not None:

                        if show_key not in self.booked_seats:
                            self.booked_seats[show_key] = set()

                        self.booked_seats[show_key].add(booking.seat)

                    number = int(booking.booking_id.replace("BK", ""))

                    if number >= self.booking_counter:
                        self.booking_counter = number + 1

            print(f"Loaded {len(self.bookings)} previous booking(s) from file.")

        except (IOError, OSError, ValueError, KeyError) as error:

            print(f"Could not load existing bookings ({error}). Starting fresh.")


    def save_booking_to_file(self, booking):

        file_exists = os.path.exists(self.FILE_NAME)

        try:

            with open(self.FILE_NAME, "a", newline="") as file:

                writer = csv.DictWriter(file, fieldnames=self.FIELD_NAMES)

                if not file_exists:
                    writer.writeheader()

                writer.writerow(booking.to_dict())

        except (IOError, OSError) as error:

            print(f"Warning: could not save booking to file ({error}).")


    def rewrite_bookings_file(self):

        try:

            with open(self.FILE_NAME, "w", newline="") as file:

                writer = csv.DictWriter(file, fieldnames=self.FIELD_NAMES)

                writer.writeheader()

                for booking in self.bookings:
                    writer.writerow(booking.to_dict())

        except (IOError, OSError) as error:

            print(f"Warning: could not update bookings file ({error}).")



    def book_ticket(self):

        movie, timing = self.select_movie_and_show()

        if movie is None:
            return

        print("\n---------- SELECTION OF DESIRED SEAT ----------")

        print("Movie:", movie.title)
        print("Show:", timing)

        selected_seat = self.select_seat(movie, timing)

        customer_name, phone = self.get_customer_details()

        ticket_type, multiplier = self.select_ticket_type()

        total_price = self.calculate_price(movie, multiplier)

        booking_id = self.generate_booking_id()

        booking_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        booking = Booking(
            booking_id,
            customer_name,
            phone,
            movie.title,
            timing,
            selected_seat,
            ticket_type,
            total_price,
            booking_time
        )

        self.bookings.append(booking)

        self.save_booking_to_file(booking)

        print("\n---------- BOOKING SUMMARY ----------")

        booking.display()

        print("\nBooking confirmed and saved successfully!")



    def view_all_bookings(self):

        print("\n---------- ALL BOOKINGS ----------")

        if not self.bookings:

            print("No bookings found yet.")

            return

        for booking in self.bookings:
            booking.display()

        print(f"\nTotal Bookings: {len(self.bookings)}")



    def cancel_booking(self):

        print("\n---------- CANCEL BOOKING ----------")

        if not self.bookings:

            print("No bookings to cancel.")

            return

        booking_id = input("Enter Booking ID to cancel: ").strip().upper()

        found_booking = None

        for booking in self.bookings:

            if booking.booking_id == booking_id:

                found_booking = booking

                break

        if found_booking is None:

            print("Booking ID not found.")

            return

        show_key = None

        for movie in self.movies:

            if movie.title == found_booking.movie_title:

                show_key = (movie.movie_id, found_booking.timing)

        if show_key is not None and show_key in self.booked_seats:

            self.booked_seats[show_key].discard(found_booking.seat)

        self.bookings.remove(found_booking)

        self.rewrite_bookings_file()

        print(f"Booking {booking_id} cancelled successfully. Seat released.")


In [7]:
def main_menu():

    bs = BookingSystem(movies)

    while True:

        print("\n==================================")
        print("     MOVIE TICKET BOOKING SYSTEM")
        print("==================================")
        print("1. View Movies")
        print("2. Search Movie")
        print("3. Book a Ticket")
        print("4. View All Bookings")
        print("5. Cancel a Booking")
        print("6. View Seat Map for a Show")
        print("7. Exit")

        choice = input("Enter your choice (1-7): ").strip()

        try:

            if choice == "1":

                bs.view_movies()

            elif choice == "2":

                bs.search_movie()

            elif choice == "3":

                bs.book_ticket()

            elif choice == "4":

                bs.view_all_bookings()

            elif choice == "5":

                bs.cancel_booking()

            elif choice == "6":

                bs.view_seat_map_menu()

            elif choice == "7":

                print("\nThank you for using the Movie Ticket Booking System. Goodbye!")

                break

            else:

                print("Invalid choice. Please select an option between 1 and 7.")

        except Exception as error:

            print(f"Something went wrong: {error}. Please try again.")


In [8]:
main_menu()


     MOVIE TICKET BOOKING SYSTEM
1. View Movies
2. Search Movie
3. Book a Ticket
4. View All Bookings
5. Cancel a Booking
6. View Seat Map for a Show
7. Exit
Enter your choice (1-7): 1

-------- movies airing -------------
ID : M101
Title : Interstellar
Genre : Sci-Fi
Duration : 169 minutes
Show Timings : ('10:00 AM', '2:30 PM', '6:25 PM')
Price : ₹200
------------------------------------------------------------
ID : M102
Title : Inception
Genre : Thriller
Duration : 148 minutes
Show Timings : ('11:00 AM', '3:15 PM', '7:00 PM')
Price : ₹180
------------------------------------------------------------
ID : M103
Title : Avengers: Infinity War
Genre : Action
Duration : 181 minutes
Show Timings : ('12:00 PM', '4:00 PM', '8:00 PM')
Price : ₹250
------------------------------------------------------------
ID : M104
Title : The Dark Knight
Genre : Action
Duration : 152 minutes
Show Timings : ('1:00 PM', '5:00 PM', '9:00 PM')
Price : ₹220
------------------------------------------------------